# Part 4 – LLM-Powered Feature

## Track C – Model Prediction Explanation Pipeline

This notebook demonstrates:
- Loading the trained model from Part 3
- Making predictions on new records
- Calling an LLM via OpenRouter API
- Validating JSON responses
- Testing PII guardrails
- Comparing temperature settings (0 vs 0.7)

## Step 1: Import Libraries

In [ ]:
import os
import json
import requests
import pandas as pd
import joblib
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print("✓ Libraries imported successfully")

## Step 2: Import Custom Modules

In [ ]:
from prompts import SYSTEM_PROMPT, USER_TEMPLATE
from utils import has_pii, validate_response, fallback
from schema import SCHEMA

print("✓ Custom modules imported")

## Step 3: Load the Model

In [ ]:
model = joblib.load("best_model.pkl")
print("✓ Model loaded successfully from best_model.pkl")

## Step 4: Set Up API Configuration

In [ ]:
OPENROUTER_API_KEY = os.getenv("LLM_API_KEY")
OPENROUTER_API_URL = "https://openrouter.ai/api/v1/chat/completions"

if not OPENROUTER_API_KEY:
    raise ValueError(
        "LLM_API_KEY not found. "
        "Please create a .env file with: LLM_API_KEY=your_key"
    )

print("✓ API key loaded from environment")

## Step 5: Define call_llm() Function

In [ ]:
def call_llm(prompt_text, temperature=0):
    """
    Call OpenRouter LLM with structured prompt.
    
    Args:
        prompt_text (str): The user prompt
        temperature (float): Controls randomness (0=deterministic, 0.7=creative)
    
    Returns:
        str: LLM response text, or None on error
    """
    
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    
    data = {
        "model": "openai/gpt-3.5-turbo",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt_text}
        ],
        "temperature": temperature,
        "max_tokens": 500,
    }
    
    try:
        response = requests.post(OPENROUTER_API_URL, headers=headers, json=data, timeout=30)
        response.raise_for_status()
        result = response.json()
        
        if "choices" in result and len(result["choices"]) > 0:
            return result["choices"][0]["message"]["content"]
        else:
            print("Error: Unexpected response format")
            return None
            
    except requests.exceptions.RequestException as e:
        print(f"API Error: {e}")
        return None

print("✓ call_llm() function defined")

## Step 6: Test LLM Connectivity

In [ ]:
print("\nTesting LLM Connectivity...")
test_prompt = "Reply with only the word: hello"
response = call_llm(test_prompt, temperature=0)

if response:
    print(f"✓ LLM is working")
    print(f"  Response: {response.strip()}")
else:
    print("✗ LLM connection failed")

## Step 7: Test PII Guardrail

In [ ]:
print("\nTesting PII Guardrail...")

# Test 1: Blocked input (email)
blocked_input = "Email: abc@gmail.com\nAge: 40"
if has_pii(blocked_input):
    print("✓ PII guardrail correctly blocked email")
else:
    print("✗ PII guardrail failed to block email")

# Test 2: Allowed input
allowed_input = "Age: 40\nBMI: 31\nSmoker: Yes"
if not has_pii(allowed_input):
    print("✓ PII guardrail correctly allowed normal input")
else:
    print("✗ PII guardrail incorrectly blocked normal input")

## Step 8: Create Test Records

In [ ]:
test_records = [
    {
        "age": 25,
        "sex_male": 1,
        "bmi": 22.5,
        "children": 0,
        "smoker_yes": 0,
        "region_northwest": 0,
        "region_southeast": 0,
        "region_southwest": 1,
    },
    {
        "age": 45,
        "sex_male": 0,
        "bmi": 31.2,
        "children": 2,
        "smoker_yes": 1,
        "region_northwest": 0,
        "region_southeast": 1,
        "region_southwest": 0,
    },
    {
        "age": 60,
        "sex_male": 1,
        "bmi": 28.0,
        "children": 3,
        "smoker_yes": 1,
        "region_northwest": 1,
        "region_southeast": 0,
        "region_southwest": 0,
    },
]

print(f"✓ Created {len(test_records)} test records")

## Step 9: Helper Functions

In [ ]:
def get_feature_names():
    """Return feature names in order."""
    return [
        "age",
        "sex_male",
        "bmi",
        "children",
        "smoker_yes",
        "region_northwest",
        "region_southeast",
        "region_southwest",
    ]

def make_prediction(model, record):
    """Make prediction on a single record."""
    feature_names = get_feature_names()
    feature_values = [[record[name] for name in feature_names]]
    
    try:
        prediction = model.predict(feature_values)[0]
        probability = model.predict_proba(feature_values)[0][1]
        return prediction, probability
    except Exception as e:
        print(f"Error making prediction: {e}")
        return None, None

def format_features_for_prompt(record):
    """Format feature values into readable text."""
    feature_text = f"""
Age: {record.get('age', 'N/A')}
Sex: {'Male' if record.get('sex_male', 0) == 1 else 'Female'}
BMI: {record.get('bmi', 'N/A')}
Children: {record.get('children', 'N/A')}
Smoker: {'Yes' if record.get('smoker_yes', 0) == 1 else 'No'}
Region: {get_region(record)}
"""
    return feature_text.strip()

def get_region(record):
    """Determine region from one-hot encoding."""
    if record.get("region_northwest", 0) == 1:
        return "Northwest"
    elif record.get("region_southeast", 0) == 1:
        return "Southeast"
    elif record.get("region_southwest", 0) == 1:
        return "Southwest"
    else:
        return "Northeast"

print("✓ Helper functions defined")

## Step 10: Run Predictions with LLM Explanations

In [ ]:
print("\n" + "="*60)
print("Running Predictions with LLM Explanations")
print("="*60)

all_results = []

for idx, record in enumerate(test_records, 1):
    print(f"\nRecord {idx}:")
    
    # Make prediction
    prediction, probability = make_prediction(model, record)
    
    if prediction is None:
        print(f"  ✗ Prediction failed")
        continue
    
    prediction_label = "High Charges" if prediction == 1 else "Low Charges"
    print(f"  Prediction: {prediction_label} (p={probability:.2%})")
    
    # Check for PII
    record_text = json.dumps(record)
    if has_pii(record_text):
        print(f"  ⚠ PII detected. Skipping LLM call.")
        continue
    
    # Format features and create prompt
    features_text = format_features_for_prompt(record)
    user_prompt = USER_TEMPLATE.format(
        features=features_text,
        prediction=prediction_label,
        probability=f"{probability:.2%}"
    )
    
    # Call LLM
    print(f"  Calling LLM...")
    response_text = call_llm(user_prompt, temperature=0)
    
    if response_text is None:
        print(f"  ✗ LLM call failed")
        result = fallback()
    else:
        result = validate_response(response_text)
    
    # Check validation
    validation_status = "PASS" if all(result.values()) else "FAIL"
    print(f"  Validation: {validation_status}")
    
    # Store result
    result["record_id"] = idx
    result["prediction"] = prediction
    result["probability"] = f"{probability:.4f}"
    all_results.append(result)

print(f"\n✓ Processed {len(all_results)} records")

## Step 11: Temperature Comparison

In [ ]:
print("\n" + "="*60)
print("Temperature Comparison")
print("="*60)

if len(test_records) > 0:
    record = test_records[0]
    prediction, probability = make_prediction(model, record)
    prediction_label = "High Charges" if prediction == 1 else "Low Charges"
    features_text = format_features_for_prompt(record)
    
    user_prompt = USER_TEMPLATE.format(
        features=features_text,
        prediction=prediction_label,
        probability=f"{probability:.2%}"
    )
    
    for temp in [0, 0.7]:
        print(f"\nTemperature: {temp}")
        response = call_llm(user_prompt, temperature=temp)
        if response:
            print(f"  Response (first 200 chars): {response[:200]}...")
            result = validate_response(response)
            print(f"  Prediction Label: {result.get('prediction_label', 'N/A')}")
            print(f"  Confidence: {result.get('confidence_level', 'N/A')}")
        else:
            print(f"  ✗ Failed to get response")

## Step 12: Save Results to CSV

In [ ]:
# Create outputs folder
outputs_folder = Path("outputs")
outputs_folder.mkdir(exist_ok=True)

# Save main results
if all_results:
    results_df = pd.DataFrame(all_results)
    results_df.to_csv(outputs_folder / "llm_results.csv", index=False)
    print(f"✓ Results saved to outputs/llm_results.csv")
    print(f"\nFirst few rows:")
    print(results_df.head())
else:
    print("No results to save")

## Step 13: Summary

In [ ]:
print("\n" + "="*60)
print("PART 4 COMPLETED SUCCESSFULLY")
print("="*60)
print("\nSummary:")
print(f"  ✓ Model loaded and tested")
print(f"  ✓ API connectivity verified")
print(f"  ✓ PII guardrail tested")
print(f"  ✓ {len(all_results)} predictions made")
print(f"  ✓ JSON validation performed")
print(f"  ✓ Temperature comparison completed")
print(f"  ✓ Results saved to outputs/")
print("\nNext steps:")
print("  1. Review outputs/llm_results.csv")
print("  2. Check validation status of each prediction")
print("  3. Compare temperature 0 vs 0.7 outputs")
print("  4. Deploy model as REST API (optional)")